# IMAS-Python 

[IMAS-Python](https://github.com/iterorganization/IMAS-Python) is the recommended Python interface for working with IMAS IDS data. It provides Python objects corresponding to each IDS defined in the [Data Dictionary](02_IMAS_Data_Dictionary.ipynb) and an API to deal with data entries that may rely on the [IMAS-Core](03_IMAS_Core.ipynb) library for I/O operations. 


## Concept and terminology 

IMAS-Python provides the following key ingredients:

- Full support of IMAS-Core, with lazy-loading;
- Dedicated APIs to interact with IDSs and with IMAS data entries;
- Ability to use all DD versions (depends on [imas-data-dictionaries](https://pypi.org/project/imas-data-dictionaries/);
- Easy access to all metadata associated with IDSs, weasy exploration and iteration over the structure;
- Ability to convert IDS between DD versions and to convert IDS to Xarray;
- Ability to store and load IDS from netCDF;
- Comes with a couple of command line tools.


## Install

IMAS-Python is a pure Python library that can be installed via pip (or [conda](https://anaconda.org/channels/conda-forge/packages/imas-python/overview), thanks to the help from the community!).

Since IMAS-Core is now also available as a package on all main platforms, it is now installed directly as a dependency of IMAS-Python. 
IMAS-Python also depends on the imas-data-dictionaries package which allows it to know all DD versions, as explained [here](02_IMAS_Data_Dictionary.ipynb#Going-further).

Note that other dependencies are optional and may be installed on-demand:

- `netcdf` enables saving IDS into netCDF [self-described format](https://imas-python.readthedocs.io/en/stable/netcdf/conventions.html#imas-conventions-for-the-netcdf-data-format) (does not use IMAS-Core);
- `xarray` enables conversion of IDS into Xarray;
- `h5py` enables the [`analyze-db`](https://imas-python.readthedocs.io/en/stable/cli.html#imas-analyze-db) command line tool 
- `saxonche` enables on-the-fly creation of models for the selected DD version when working with mdsplus backend (requires install of [IMAS-Core from sources](03_IMAS_Core.ipynb#Distribution-package-limitations));
- `test` installs all dependencies required to run `pytest`.


In [ ]:
# this cell has some tricks that are relevant only for running in notebooks, you probably don't need these elsewhere
#import os
#import sysconfig
#os.environ["PATH"] += sysconfig.get_path('scripts')

In [ ]:
# install the latest release of the package with pip, with options to be able to use netcdf and xarray
!pip install imas-python[netcdf]

# let's verify that we can use the command line tool 
!imas --help

# and check which versions we got
!imas version

### Default DD version

By default IMAS-Python will use the latest release of the Data Dictionary as it's default target version when creating new IDSs, when opening/creating a data-entry, when reading IDS data with [`autoconvert=True`](#Advanced-tricks).

Note that while the API allows you to select which version you target for each operation (and by consequence not use the default one) you also have the option to overwrite this default by setting the `IMAS_VERSION` environement variable.

In [ ]:
# verifying that IMAS-Python picks up $IMAS_VERSION as its default DD version
!IMAS_VERSION=3.42.0 imas version | grep Default

## Creating IDS from scratch

Now let's start using the IMAS-Python to structure some data into an IDS form. 

### The `IDSFactory`

When creating new IDS objects we start from the [`IDSFactory` API](https://imas-python.readthedocs.io/en/stable/generated/imas.ids_factory.IDSFactory.html#imas.ids_factory.IDSFactory), which allows to decide which version of the DD we want to use, and will expose the corresponding list of IDS and their internal structures.


In [ ]:
# first we import the imas package (and numpy for making up some mock data)
import imas

# then we create an IDSFactory for our chosen version of the DD 
ids3 = imas.IDSFactory(version="3.42.0")
# or for the latest (the default)
ids = imas.IDSFactory()

# we can verify which DD version is associated with this factory, and what is the list of IDS available in this version
print(f"\nThe Factory is in DD version {ids.version}")
print(f"The following IDSs are defined: {ids.ids_names()}")


### Instantiate an IDS object

In [ ]:
# we can now instantiate an IDS object
cp = ids.core_profiles()
# or from a string
cp = ids.new("core_profiles")


### Explore with `inspect`

We will also use some utility functions from [`imas.util`](https://imas-python.readthedocs.io/en/stable/generated/imas.util.html#module-imas.util) to simplify our life when exploring the various objects: first the `inspect` function and later the `print_tree` function.

In [ ]:
from imas.util import inspect, print_tree

# let's use the imas.util.inspect function to explore 
print("\nTop level structure of the core_profiles IDS")
inspect(cp)

# we can get more details from metadata associated with each object
print("\nAll the metadata associated with a field are available. Most of them are defined by the Data Dictionary, a few others are just IMAS-Python own internal use.")
inspect(cp.global_quantities.t_e_volume_average.metadata)
print("\nSub-structures (nodes of the IDS) also contain useful metadata.")
inspect(cp.profiles_1d.metadata)


### Fill data in IDS object

In [ ]:
import numpy as np

# we found from the metadata that profiles_1d is a dynamic (i.e. time-dependent) array of structure (STRUCT_ARRAY)
# let's fill some mock profiles data in this structure for a single "time slice" 
rho = np.linspace(0, 1, 50)

cp.time = [0.]
cp.profiles_1d.resize(1)
cp.profiles_1d[0].electrons.temperature = 10000 * (1 - rho**2) 
# you can also use a dict-like accessor
cp[f"profiles_1d[0]/electrons/density"] = 5e19 * (1 - 0.8*rho**2) 
# let's not forget to store the associated coordinate for these quantities
cp.profiles_1d[0].grid.rho_tor_norm = rho

# now for storing time we have two solutions depending on the value we set in homogeneous_time 
print(f"We must set homogeneous_time, here is the description of this node:\n")
print(cp.ids_properties.homogeneous_time.metadata.documentation)
cp.ids_properties.homogeneous_time = 1 


### Print the content of the IDS

In [ ]:
# let's summarize what we stored so far in this core_profiles IDS using another function from imas.util
print_tree(cp)
# note that ids_properties/version_put is filled in automatically for provenance purpose

## Storing IDSs 

So far we created an IDS object in our memory space, to make it persistent we want now to store it on disk in a data entry that can contain several related IDSs that describes my simulation or experiment.

### The `DBentry`

Creating a new [`DBEntry`](https://imas-python.readthedocs.io/en/stable/generated/imas.db_entry.DBEntry.html) requires two mandatory arguments: 

- a [`uri`](https://imas-python.readthedocs.io/en/stable/generated/imas.db_entry.DBEntry.html#imas.db_entry.DBEntry.__init__.uri) to specify information about the resource associated with the data entry (here we are going to test both a netCDF file and the HDF5 backend of IMAS-Core);
<div class="alert alert-block alert-info">
<b>Info:</b> Note that the netCDF format works slightly differently compared to IMAS-Core's data entries (associated to a folder), in netCDF's case all data are always stored in a single .nc file, so the API only contains the path to this file.</div>

- a [`mode`](https://imas-python.readthedocs.io/en/stable/generated/imas.db_entry.DBEntry.html#imas.db_entry.DBEntry.__init__.mode) to decide how to open this data entry (`w` will create the file, overwriting it in case it exists already).


### The `put` method

To store and an IDS into this DBEntry object, we call its [`put` method](https://imas-python.readthedocs.io/en/stable/generated/imas.db_entry.DBEntry.html#imas.db_entry.DBEntry.put). Be aware that this method method always clears existing data for this IDS if present in the data entry (with the [`delete_data` method](https://imas-python.readthedocs.io/en/stable/generated/imas.db_entry.DBEntry.html#imas.db_entry.DBEntry.delete_data)). In addition, it also checks that the coordinates associated with its nD array quantities are consistent (with the [`validate` method](https://imas-python.readthedocs.io/en/stable/generated/imas.ids_toplevel.IDSToplevel.html#imas.ids_toplevel.IDSToplevel.validate)).


### Using the netCDF backend 

A given `DBEntry` instance is associated with a given backend or method for storing the data. Here we are using the netCDF backend which comes as an option for IMAS-Python and does not rely on IMAS-Core. In the example below we are also using the DBEntry as a context manager so we don't have to care with closing it when we stop operations on it.


In [ ]:
# let's create a DBEntry for netCDF using context manager 
print(f"DBEntry can be used as a context manager")
with imas.DBEntry("./mycoreprofiles.nc","w") as nc:
    print(f"A DBEntry is also associated with a DD version, here the default {nc.dd_version}")
    # now let's save our core_profiles IDS into this data entry
    nc.put(cp)

# here the nc DBEntry is not opened anymore, but we created a netCDF file that contains our data
!ls -l ./mycoreprofiles.nc


### Use IMAS-Python command line tool to print IDS 

In [ ]:
!imas print --help
!imas print ./mycoreprofiles.nc core_profiles

## The `put_slice` method

One may want to store new time slices of IDS data progressively in the file, for instance while running a long time-evolving simulation.
While this is not supported by the [netCDF DBEntry](https://imas-python.readthedocs.io/en/stable/netcdf.html#implemented-features-of-a-netcdf-dbentry), some of IMAS-Core backends support this capability via the [`put_slice` method](https://imas-python.readthedocs.io/en/stable/generated/imas.db_entry.DBEntry.html#imas.db_entry.DBEntry.put_slice).

<div class="alert alert-block alert-info">
<b>Info:</b> The `put_slice` method will append one or several time slices after any existing ones on disk, it will store only time-dependent data (unless if the IDS is empty on disk, in such case it will also store the time-independent data during the first call). Time ordering is the responsibility of the user and shall monotonically increase.</div>

### Using an IMAS-Core backend (e.g. HDF5)


In [ ]:
# let's create a DBentry for HDF5
dentry = imas.DBEntry("imas:hdf5?path=./mytestpulse","w")
# this dentry object can be used for I/O operations until it we call its `close()` method
inspect(dentry)

# first we put the same initial slice with data at time=0.0, in our dentry from a few cells above
dentry.put(cp)

# then let's define a few extra time points
time = [0.1, 0.2, 0.3]
# we resize without "keeping" existing profiles_1d data in the IDS object in memory, as we already have it stored on disk
cp.profiles_1d.resize(len(time),keep=False)
for i in range(len(time)):
    cp.profiles_1d[i].electrons.temperature = 10000 * (1 - rho**2) * (1 + 0.2*time[i])
    cp[f"profiles_1d[{i}]/electrons/density"] = 5e19 * (1 - 0.8*rho**2) * (1 + 0.1*time[i])
    cp.profiles_1d[i].grid.rho_tor_norm = rho
cp.time = time

# we can now append the extra slices into the existing dentry with the put_slice method
dentry.put_slice(cp)
# now we can close this dentry if we have finished our operations on it
dentry.close()

!imas print imas:hdf5?path=./mytestpulse core_profiles

## Loading IDSs

Loading an IDS (or a part of it) from a given data entry also starts by instantiating a `DBEntry` object for the corresponding backend, but remind that the `w` mode we used previously is for creating a new data entry so it will overwrite any existing one, here instead we are going to use `r` (check [` mode`](https://imas-python.readthedocs.io/en/stable/generated/imas.db_entry.DBEntry.html#imas.db_entry.DBEntry.__init__.mode) for more options).


### The `get` method and lazy access

To read the entire IDS data, including all time points, you can use the [`get` method](https://imas-python.readthedocs.io/en/stable/generated/imas.db_entry.DBEntry.html#imas.db_entry.DBEntry.get).  In case you may not be interested in the entire structure, you may use the [`lazy`](https://imas-python.readthedocs.io/en/stable/generated/imas.db_entry.DBEntry.html#imas.db_entry.DBEntry.get.lazy) loading option.

<div class="alert alert-block alert-info">
<b>Info:</b> Note that the version of the DD may differ between the current default version (or your chosen one via the <code>IMAS_VERSION</code> environment variable or via the <a href=https://imas-python.readthedocs.io/en/stable/generated/imas.db_entry.DBEntry.html#imas.db_entry.DBEntry.__init__.dd_version><code>dd_version</code></a> argument of the <code>DBEntry</code>) and the version of the IDSs in the data entry on file. The <code>get</code> method will try to convert automatically the IDS to the targeted DD version, but you may want to keep the source DD version by passing the extra argument <a href=https://imas-python.readthedocs.io/en/stable/generated/imas.db_entry.DBEntry.html#imas.db_entry.DBEntry.get.autoconvert><code>autoconvert=False</code></a>.</div>


In [ ]:
# let's open our previous data entry for reading
with imas.DBEntry("imas:hdf5?path=./mytestpulse","r") as din:
    # reading the IDS in lazy loading mode returns instantaneously 
    cp = din.get("core_profiles",lazy=True)
    # data will be read only when being accessed in the IDS object
    print(f"time = {cp.time}")
    print(f"T_e for one to last time = {cp.profiles_1d[-2].electrons.temperature}")
    print(f"T_e for last time = {cp.profiles_1d[-1].electrons.temperature}")


### Extracting a single time slice with the `get_slice` method

We may be interested in a single point in time (which may or not exists in the data entry), so instead of reading the entire data as previously we can use the [`get_slice` method](https://imas-python.readthedocs.io/en/stable/generated/imas.db_entry.DBEntry.html#imas.db_entry.DBEntry.get_slice) instead. This method requires two mandatory arguments: the requested time value and the [`interpolation_method`](https://imas-python.readthedocs.io/en/stable/generated/imas.db_entry.DBEntry.html#imas.db_entry.DBEntry.get_slice.interpolation_method) to be applied if the time value is not present in this IDS.

<div class="alert alert-block alert-info">
<b>Info:</b> If you are interested in several time points, or time samples at a given frequency you can use the <a href=https://imas-python.readthedocs.io/en/stable/generated/imas.db_entry.DBEntry.html#imas.db_entry.DBEntry.get_sample><code>get_sample</code> method</a>. Both   <code>get_slice</code> and <code>get_sample</code> are only available with IMAS-Core, not with netCDF.</div> 


In [ ]:
# useful constant parameters are available in imas.ids_defs
from imas.ids_defs import LINEAR_INTERP

# let's open our previous time-dependent data entry
with imas.DBEntry("imas:hdf5?path=./mytestpulse","r") as din:
    # reading a single slice, interpolating linearly the values at the requested time point
    cp = din.get_slice("core_profiles",time_requested=0.25,interpolation_method=LINEAR_INTERP)
    # data is calculated by interpolating linearly between the two adjacent times
    print(f"T_e for time = {cp.time[0]}: {cp.profiles_1d[0].electrons.temperature}")


## Additional Resources

- [IMAS-Python Repository](https://github.com/iterorganization/IMAS-Python)
- [IMAS-Python Documentation](https://imas-python.readthedocs.io/en/stable/)
- [IMAS-Python Training Examples](https://imas-python.readthedocs.io/en/stable/courses/basic_user_training.html)

---
[<b>&larr;</b>IMAS-Core](03_IMAS_Core.ipynb) &nbsp;&nbsp;&nbsp; [<b>&uarr;</b>Tutorial TOC](00_Start_Here.ipynb) &nbsp;&nbsp;&nbsp; [IDStools<b>&rarr;</b>](04_IDStools.ipynb)